# Experiment 1 — Baseline

**EEEM068 Applied Machine Learning — University of Surrey**

Baseline run with **no class weighting, no weighted sampling, and default hyperparameters**.  
This establishes the starting point against which all subsequent experiments are measured.

| Setting | Value |
|---|---|
| Architecture | EfficientNet-B0 (ImageNet pretrained) |
| Loss | CrossEntropyLoss (no class weights) |
| Sampler | Standard shuffle |
| Augmentation | Resize → RandomCrop → HFlip → Normalise |
| Optimiser | AdamW, lr=3e-4, weight_decay=1e-2 |
| Scheduler | CosineAnnealingLR |
| Metric | Quadratic Weighted Kappa (QWK) |

## 1. Imports

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
from torchvision import transforms
from sklearn.metrics import (
    classification_report, cohen_kappa_score, confusion_matrix
)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

PyTorch : 2.11.0+cu130
CUDA    : True


## 2. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
NOTEBOOK_DIR    = Path().resolve()
ROOT            = NOTEBOOK_DIR.parent

DATA_ROOT       = ROOT / 'data' / 'diabetic-retinopathy-detection'
TRAIN_DIR       = DATA_ROOT / 'train'
VAL_DIR         = DATA_ROOT / 'val'
TEST_LABELS_DIR = DATA_ROOT / 'test_labels'
TRAIN_CSV       = DATA_ROOT / 'train_split.csv'
VAL_CSV         = DATA_ROOT / 'val_split.csv'
TEST_CSV        = DATA_ROOT / 'test_split.csv'

OUT_DIR     = ROOT / 'outputs' / 'baseline'
CKPT_DIR    = OUT_DIR / 'checkpoints'
FIGURES_DIR = OUT_DIR / 'figures'
for d in (OUT_DIR, CKPT_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Experiment ────────────────────────────────────────────────
EXPERIMENT_NAME = 'baseline'

# ── Reproducibility ───────────────────────────────────────────
SEED = 42

# ── Data ──────────────────────────────────────────────────────
IMAGE_SIZE   = 224
NUM_CLASSES  = 5
CLASS_NAMES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# ── Training ──────────────────────────────────────────────────
BATCH_SIZE               = 32
NUM_WORKERS              = 4
EPOCHS                   = 10
LR                       = 3e-4
WEIGHT_DECAY             = 1e-2
EARLY_STOPPING_PATIENCE  = 3

print('Config set. Root:', ROOT)

Config set. Root: /scratch/AML_G14_CW


In [3]:
# ── Seed + device ─────────────────────────────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## 3. Load Pre-Split Data

In [ ]:
# Load the 70/15/15 stratified split created in data_preprocessing.ipynb
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

print(f'Train : {len(train_df):,} images  (dir: train/)')
print(f'Val   : {len(val_df):,} images   (dir: val/)')
print(f'Test  : held out — loaded only in Section 11 for final evaluation')
print()
print(f'{"Grade":<6} {"Name":<18} {"Train":>7} {"Val":>7}')
print('-' * 42)
for g, name in enumerate(CLASS_NAMES):
    t = (train_df['level'] == g).sum()
    v = (val_df['level'] == g).sum()
    print(f'  {g}    {name:<18} {t:>6,}  {v:>6,}')

## 4. Dataset and DataLoaders

In [7]:
class DRDataset(Dataset):
    """Fundus image dataset. Reads images from data_dir using the 'image' column."""

    def __init__(self, df: pd.DataFrame, data_dir: Path, transform=None):
        self.df       = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        image = Image.open(self.data_dir / f"{row['image']}.jpeg").convert('RGB')
        label = int(row['level'])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Minimal augmentation — baseline should not benefit from heavy augmentation
train_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = DRDataset(train_df, TRAIN_DIR, transform=train_transform)
val_ds   = DRDataset(val_df,   VAL_DIR,   transform=val_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'Train loader: {len(train_loader)} batches/epoch')
print(f'Val loader  : {len(val_loader)} batches/epoch')

Train loader: 768 batches/epoch
Val loader  : 330 batches/epoch


## 5. Model

In [9]:
# EfficientNet-B0 with ImageNet weights, 5-class head
model = tv_models.efficientnet_b0(weights='DEFAULT')
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable:,}')

Total params    : 4,013,953
Trainable params: 4,013,953


## 6. Loss, Optimiser, Scheduler

Plain `CrossEntropyLoss` with no class weights — this is the unmodified baseline.

In [10]:
criterion = nn.CrossEntropyLoss()   # no class weights
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Loss      :', criterion)
print('Optimiser :', optimizer.__class__.__name__, f'lr={LR}')
print('Scheduler :', scheduler.__class__.__name__)

Loss      : CrossEntropyLoss()
Optimiser : AdamW lr=0.0003
Scheduler : CosineAnnealingLR


## 7. Training Loop

In [ ]:
use_amp = device.type == 'cuda'
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)


def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc='train' if train_mode else 'val'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, labels)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        total      += labels.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / total, all_labels, all_preds


print('run_epoch defined.')

In [ ]:
best_path     = CKPT_DIR / f'{EXPERIMENT_NAME}_best.pt'
best_val_loss = float('inf')
patience_left = EARLY_STOPPING_PATIENCE

history = {'train_loss': [], 'val_loss': []}

print(f"{'Epoch':<8} {'Train Loss':>12} {'Val Loss':>12}")
print('─' * 35)

for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _ = run_epoch(train_loader, train_mode=True)
    vl_loss, _, _ = run_epoch(val_loader,   train_mode=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)

    marker = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_left = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, best_path)
        marker = '  ← best'
    else:
        patience_left -= 1

    print(f'{epoch:<8} {tr_loss:>12.4f} {vl_loss:>12.4f}{marker}')

    if patience_left == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val loss: {best_val_loss:.4f}  →  {best_path}')

## 8. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_ran, history['train_loss'], label='Train Loss', marker='o', markersize=5)
ax.plot(epochs_ran, history['val_loss'],   label='Val Loss',   marker='s', markersize=5)
ax.set_title('Baseline — Train vs Validation Loss', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / f'{EXPERIMENT_NAME}_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/baseline/{EXPERIMENT_NAME}_loss_curve.png')


## 9. Evaluation on Validation Set

In [ ]:
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}  (val loss = {ckpt["val_loss"]:.4f})')

_, y_true, y_pred = run_epoch(val_loader, train_mode=False)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print(f'Quadratic Weighted Kappa: {cohen_kappa_score(y_true, y_pred, weights="quadratic"):.4f}')


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=ax)

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — Baseline', fontweight='bold')

thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm[i, j]:,}',
                ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(FIGURES_DIR / f'{EXPERIMENT_NAME}_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/baseline/{EXPERIMENT_NAME}_confusion_matrix.png')

## 10. Baseline Summary

Record these results before moving to the next experiment.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

qwk    = cohen_kappa_score(y_true, y_pred, weights='quadratic')
acc    = accuracy_score(y_true, y_pred)
f1_per = f1_score(y_true, y_pred, average=None, labels=list(range(5)))

print('=' * 50)
print(f'  EXPERIMENT 1 — BASELINE RESULTS')
print('=' * 50)
print(f'  Val QWK      : {qwk:.4f}')
print(f'  Val Accuracy : {acc:.4f}')
print()
print(f'  Per-class F1:')
for name, f1 in zip(CLASS_NAMES, f1_per):
    print(f'    {name:<18}: {f1:.4f}')
print('=' * 50)

## 11. Final Evaluation on Test Set

> **Run this section only once — after all experiments are complete and a final model has been selected.**  
> The test set must not be used to guide any training or hyperparameter decisions.  
> Images live in `test_labels/`, labels in `test_split.csv`.

In [ ]:
test_df = pd.read_csv(TEST_CSV)

test_ds = DRDataset(test_df, TEST_LABELS_DIR, transform=val_transform)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'Test set : {len(test_df):,} images  ({len(test_loader)} batches)')
print()
print(f'{"Grade":<6} {"Name":<18} {"Count":>7}')
print('-' * 34)
for g, name in enumerate(CLASS_NAMES):
    print(f'  {g}    {name:<18} {(test_df["level"]==g).sum():>6,}')

In [ ]:
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (val loss = {ckpt["val_loss"]:.4f})')

_, test_y_true, test_y_pred = run_epoch(test_loader, train_mode=False)

test_qwk = cohen_kappa_score(test_y_true, test_y_pred, weights='quadratic')
test_acc  = accuracy_score(test_y_true, test_y_pred)

print('\nClassification Report (Test Set):')
print(classification_report(test_y_true, test_y_pred, target_names=CLASS_NAMES, digits=3))

print('=' * 50)
print('  BASELINE — FINAL TEST RESULTS')
print('=' * 50)
print(f'  Test QWK      : {test_qwk:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}')
print()
print('  Per-class F1:')
test_f1 = f1_score(test_y_true, test_y_pred, average=None, labels=list(range(5)))
for name, f1 in zip(CLASS_NAMES, test_f1):
    print(f'    {name:<18}: {f1:.4f}')
print('=' * 50)

In [ ]:
cm_test = confusion_matrix(test_y_true, test_y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_test, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Baseline (Test Set)', fontweight='bold')

thresh = cm_test.max() / 2
for i in range(cm_test.shape[0]):
    for j in range(cm_test.shape[1]):
        ax.text(j, i, f'{cm_test[i, j]:,}', ha='center', va='center', fontsize=9,
                color='white' if cm_test[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(FIGURES_DIR / f'{EXPERIMENT_NAME}_test_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/baseline/figures/{EXPERIMENT_NAME}_test_confusion_matrix.png')